# Смесь долей Session6 и CatBoost depth 4

Запуск на CPU. Обучается только метамодель на готовых базовых OOF. На каждой неделе подбора CatBoost видит только предыдущие недели. Сравниваем четыре веса при одном объёме Session6 для каждой пары «день × горизонт».

Вес выбираем по среднему дневному score на двух неделях подбора. При равенстве выбираем меньший вес CatBoost. После фиксации обучаем метамодель на всех четырёх OOF-неделях и проверяем выбранную смесь, чистую Session6 и чистую метамодель на локальном test. Последние два варианта служат контролями.

Локальный test оценивается до финального переобучения. Этот период уже изучался. Ноутбук сохраняет диагностические прогнозы и метрики. Для конкурсного сабмита нужен отдельный полный refit.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path(os.environ.get('BLEND_PACKAGE', '/root/experiments/session6_blend_server'))
OOF_PATH = None
TEST_PATH = None
OUT = ROOT / 'output_shape_shrink_cpu'
OUT.mkdir(parents=True, exist_ok=True)

BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
GROUP_KEYS = ['cutoff', 'horizon']
KEYS = GROUP_KEYS + ['route_id']
COLUMNS = KEYS + BASES + ['y', 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
THREADS = 2
SEED = 42

def read_panel(path, name, start, end):
    archived = ROOT / 'blend_inputs' / 'blend_best_output' / name
    if path is None and archived.is_file():
        path = archived
    if path is None:
        candidates = [p for p in ROOT.rglob(name) if OUT not in p.parents
                      and set(COLUMNS).issubset(pq.ParquetFile(p).schema.names)]
        if len(candidates) != 1:
            raise ValueError(f'Укажите путь явно для {name}: {candidates}')
        path = candidates[0]
    frame = pd.read_parquet(path, columns=COLUMNS)
    frame['cutoff'] = pd.to_datetime(frame.cutoff)
    expected = pd.date_range(start, end, inclusive='left') + pd.Timedelta(hours=10, minutes=30)
    actual = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    counts = frame.groupby(GROUP_KEYS).size()
    if (not actual.equals(expected) or frame.duplicated(KEYS).any()
            or set(frame.horizon.unique()) != set(range(1, 9))
            or len(counts) != len(expected)*8 or not counts.eq(1000).all()
            or not np.isfinite(frame[COLUMNS[3:]].to_numpy()).all()
            or frame.y.lt(0).any() or frame.session6.lt(0).any()):
        raise ValueError(f'Некорректная панель: {path}')
    print(path, len(frame))
    return frame.reset_index(drop=True)


In [ ]:
def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level

def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def fit_meta(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix, frame.y)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    model = CatBoostRegressor(depth=4, iterations=220, loss_function='MAE', learning_rate=.035,
                              l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                              task_type='CPU', thread_count=THREADS)
    model.fit(features, (frame.y.to_numpy(dtype=float)-baseline)/level, sample_weight=level/level.mean())
    return model, weights

def predict_meta(frame, fitted):
    model, weights = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    return with_volume(frame, baseline + np.clip(model.predict(features), -.5, .5)*level)

def score_mix(frame, shape, weight):
    prediction = with_volume(frame, (1-weight)*frame.session6.to_numpy() + weight*shape)
    daily = pd.DataFrame([dict(day=str(day.date()), **metric(group.y, prediction[group.index]))
                          for day, group in frame.groupby('cutoff')])
    summary = dict(**metric(frame.y, prediction), daily_mean=daily.score.mean(),
                   daily_p90=daily.score.quantile(.9), daily_worst=daily.score.max())
    return summary, daily, prediction


In [ ]:
oof = read_panel(OOF_PATH, 'base_oof.parquet', '2025-09-20', '2025-10-18')
weights_to_check = [0., .5, .75, 1.]
validation_rows = []
validation_daily = []
train_rows = []
for boundary, end in [('2025-10-04','2025-10-11'), ('2025-10-11','2025-10-18')]:
    train = oof.loc[oof.cutoff < boundary].reset_index(drop=True)
    val = oof.loc[(oof.cutoff >= boundary) & (oof.cutoff < end)].reset_index(drop=True)
    fitted = fit_meta(train)
    train_shape = predict_meta(train, fitted)
    val_shape = predict_meta(val, fitted)
    for weight in weights_to_check:
        train_metrics, _, _ = score_mix(train, train_shape, weight)
        val_metrics, daily, _ = score_mix(val, val_shape, weight)
        train_rows.append(dict(week=boundary, weight=weight, **train_metrics))
        validation_rows.append(dict(week=boundary, weight=weight, **val_metrics))
        validation_daily.append(daily.assign(week=boundary, weight=weight))

validation = pd.DataFrame(validation_rows)
ranking = validation.groupby('weight', as_index=False).daily_mean.mean().sort_values(['daily_mean','weight'])
winner = float(ranking.iloc[0].weight)
pd.DataFrame(train_rows).to_csv(OUT/'train_by_fold.csv',index=False)
validation.to_csv(OUT/'validation.csv',index=False)
pd.concat(validation_daily,ignore_index=True).to_csv(OUT/'validation_daily.csv',index=False)
(OUT/'frozen_selection.json').write_text(json.dumps(dict(weight=winner, criterion='mean daily score on pretest folds'),indent=2))
display(pd.DataFrame(train_rows))
display(validation)
display(ranking)
print('Выбранный вес CatBoost:', winner)


In [ ]:
test = read_panel(TEST_PATH,'test_predictions.parquet','2025-10-18','2025-11-01')
fitted = fit_meta(oof)
rows = []
daily_rows = []
predictions = test[KEYS + ['y']].copy()
for period, frame in [('train_oof',oof), ('test',test)]:
    shape = predict_meta(frame,fitted)
    for variant, weight in [('session6',0.), ('selected',winner), ('meta',1.)]:
        values, daily, pred = score_mix(frame,shape,weight)
        rows.append(dict(period=period,variant=variant,weight=weight,**values))
        daily_rows.append(daily.assign(period=period,variant=variant))
        if period == 'test':
            predictions[variant] = pred
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT/'metrics.csv',index=False)
pd.concat(daily_rows,ignore_index=True).to_csv(OUT/'daily.csv',index=False)
predictions.to_parquet(OUT/'test_predictions.parquet',index=False)
display(metrics)
print('train_oof — оценка на данных обучения метамодели. Test оценён до финального переобучения.')
print('Результаты:', OUT)
